# CDC — three delivery routes, one catalog

The CDC publishes mortality and health-behaviour statistics, and meida serves
them through three different doors. `data.cdc.gov` (Socrata) is a live JSON
API, so its series are fetched per request. The NVSR life tables are published
only as PDF reports, and CDC WONDER is throttled to roughly one query every two
minutes behind a bot filter — neither can be fetched while a caller waits, so
both are parsed once and stored in Postgres.

The surprising part is that the route is a property of the *series*, not of the
question. Ask for US life expectancy at birth and you get two answers: a
Socrata series ending in 2018 and an NVSR series starting in 2021, disjoint
except that neither covers the other's years. So the useful first move is never
"which API?" but "which series exist?" — and `series_catalog_search` answers
that across all three routes, each result carrying a `retrieval` block naming
the tool that fetches it.

`notebooks/cdc/mcp.ipynb` is the long-form reference for these tools; this is
the short tour. The MCP server must be running.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
from matplotlib import pyplot
from lib import config

sys.path.append('../')

from utils import (
    list_mcp_tools,
    show_tool_schema,
    catalog_concepts,
    search_catalog,
    get_rows,
    get_stored_series,
    plot_cdc_series,
    plot_stored_series,
)

pyplot.style.use(config.glyfish_style)

## 1. Discovery

The server carries every source, so CDC is three tool prefixes rather than one.
`cdc_*` wraps the live Socrata API. `timeseries_source_*` is deliberately
generic — the source is an argument, not part of the tool name, so WONDER and
NVSR (and the next file-delivered source) need no new tools. `series_catalog_*`
sits above both and is the only place the two routes are described in the same
vocabulary.

In [ ]:
await list_mcp_tools(prefix=('cdc_', 'timeseries_source_', 'series_catalog_'))

The two fetch tools read very differently, and the schemas say why.

`cdc_series_data` selects a series by **naming its facets** — `state`, `race`,
`sex`, `drug` — never by SoQL. The server owns the column names and literals,
which is what lets one tool span datasets that spell the same facet three
different ways. The catch is that its enums are the union across every dataset,
so a plausible-looking pair like `age` + `race` may describe a series nobody
published; the catalog, not the schema, is what says which combinations exist.

`timeseries_source_data` takes only `source` and `native_id`. There are no
facets to name because a stored series had its facets resolved into an id when
it was loaded.

In [ ]:
_ = await show_tool_schema('cdc_series_data')
print()
_ = await show_tool_schema('timeseries_source_data')

## 2. Finding a series

`series_catalog_concepts` is the coarse map: what is measured, and where. A row
with a dataset id is live Socrata; a row with `-` is stored, because a stored
series has no Socrata dataset behind it.

Two details make search worth trusting. `total` counts every match *before*
`limit` is applied, so `total != returned` is how a caller detects truncation
rather than concluding the data is thin. And `search_catalog` rejects unknown
keyword arguments outright: an unrecognised name would otherwise be forwarded
as a filter on a facet nobody publishes, which matches nothing and reads like
"no such data" instead of "you misspelled an argument".

In [ ]:
_ = await catalog_concepts()

In [ ]:
# 8 series match, 5 come back: the counts differ exactly when limit truncated
_ = await search_catalog(concept='drug_overdose', state='OH', limit=5)

# a mistyped facet is refused rather than sent as a filter matching nothing
try:
    await search_catalog(concept='drug_overdose', states='OH')
except TypeError as error:
    print(f'\nTypeError: {error}')


In [ ]:
# one question, two routes. Each entry names its own fetch tool, so a client
# can dispatch on `retrieval` without knowing anything about CDC's plumbing.
life = await search_catalog(concept='life_expectancy', race='all', sex='both')
routes = {entry['retrieval']['tool']: entry for entry in life}

for tool, entry in routes.items():
    print(f"\n{tool}"
          f"\n  {entry['title']}"
          f"\n  {entry['observation_start'][:4]}..{entry['observation_end'][:4]}"
          f" in {entry['units']}"
          f"\n  retrieval={entry['retrieval']}")


## 3. Fetch and plot

### Socrata — fetched live

The entry's `facets` are exactly the keyword arguments `cdc_series_data` takes,
so the row read off the search feeds straight back into the fetch with no
translation step — that symmetry is the point of naming facets instead of
writing queries.

What comes back is the century-long series: life expectancy at birth climbing
from 47 years in 1900, with the 1918 influenza pandemic cut into it as a
single-year collapse of more than a decade, deeper than anything since.

In [ ]:
socrata = routes['cdc_series_data']
print(f"facets {socrata['facets']} -> cdc_series_data arguments")

rows = await get_rows(socrata['dataset_id'], socrata['concept'], **socrata['facets'])
print(f"{len(rows)} rows, e.g. {rows[0]} .. {rows[-1]}")

plot_cdc_series(rows, 'year', 'value',
                title=f"{socrata['title']} — CDC {socrata['dataset_id']}",
                ylabel=socrata['units'])


### NVSR — stored life tables

The same measurement, taken over where Socrata stops. These four years are the
only channel publishing US life expectancy by race after 2020: WONDER has none,
and the figures exist publicly only inside NVSR report PDFs. Because they are
parsed from the life tables rather than the press release, they keep full
precision — `78.971`, not the rounded `79.0` the reports lead with.

Four annual points is a thin plot, and that is the honest picture of this
route: it is narrow, recent and irreplaceable.

In [ ]:
retrieval = routes['timeseries_source_data']['retrieval']
nvsr = await get_stored_series(retrieval['source'], retrieval['native_id'])

print(f"{nvsr['title']}"
      f"\n  {nvsr['units']} | {nvsr['observation_count']} obs | stale={nvsr['stale']}")
for obs in nvsr['observations']:
    print(f"  {obs['date'][:4]}  {obs['value']}")

plot_stored_series(nvsr)

### WONDER — stored cause-of-death rates

The third route, reached the same way: search a concept, follow the `retrieval`
block. WONDER's series are death-certificate rates age-adjusted to the 2000
standard population, stitched across two databases (D76 for 1999–2020, D158 for
2018–2024) that agree exactly on their overlap.

`despair_composite` is queried as the *union* of the alcohol-induced,
drug-induced and suicide code sets rather than as their sum, because those code
sets overlap and adding the three series would count the shared codes twice.
Read `excluded_codes` before quoting the numbers: WONDER refuses the NCHS
pseudo-codes, so terrorism-reclassified deaths are absent by construction.

In [ ]:
wonder_hits = await search_catalog(concept='despair_composite')
retrieval = wonder_hits[0]['retrieval']

wonder = await get_stored_series(retrieval['source'], retrieval['native_id'])
meta = wonder['metadata']['cdc_wonder']
print(f"\n{wonder['title']}"
      f"\n  {wonder['units']} | {wonder['observation_count']} obs"
      f" | databases {meta['databases']}"
      f"\n  excluded codes: {meta['excluded_codes']}")

plot_stored_series(wonder, title='Deaths of despair, age-adjusted, 1999-2024 (CDC WONDER)')